In [ ]:
# Copyright (c) Qualcomm Innovation Center, Inc.
# All rights reserved
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

# Hugging Face Transformer Model → Qualcomm NPU

## 1. Configure

In [ ]:
from executorch.backends.qualcomm.export_utils import (
    get_backend_type,
    make_quantizer,
)
from executorch.backends.qualcomm.hf_transformers.api import get_quant_recipe
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from transformers.exporters import ExecutorchExporter, ExecutorchQnnConfig

backend_hardware = "htp"
soc_models = ["SM8750", "SC8380XP"]  # Target SoCs for exported model
model_id = "NousResearch/Llama-3.2-1B"
max_seq_len = 32
sample_inputs = "Simply put, the theory of relativity states that"

## 2. Load from the Hub

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype="float32",
    attn_implementation="eager",
    generation_config=GenerationConfig(
        use_cache=True,
        cache_implementation="static",
        max_length=max_seq_len,
        cache_config={"batch_size": 1, "max_cache_len": max_seq_len},
    ),
).eval()
tokenizer = AutoTokenizer.from_pretrained(model_id)

## 3. Load calibration dataset

In [ ]:
with open("calibration_dataset.txt", "r") as f:
    dataset = f.read()

calibration_dataset = [tokenizer.encode(dataset)]

## 4. Set quantizer

In [ ]:
quantizer = make_quantizer(
    backend=get_backend_type(backend_hardware), soc_model=soc_models
)

quantizer.set_recipe(get_quant_recipe(model_id, verbose=False).recipe)


# Alternative: skip the recipe and apply a single config to the whole model.
#
# from executorch.backends.qualcomm.quantizer.quantizer import QuantDtype
# quantizer.set_default_quant_config(quant_dtype=QuantDtype.use_8a8w)

## 5. Export

One call: prepare, quantize, compile.

In [ ]:
et_program_manager = ExecutorchExporter().export(
    model=model,
    sample_inputs=tokenizer.encode(sample_inputs),
    config=ExecutorchQnnConfig(
        soc_models=soc_models,
        calibration_dataset=calibration_dataset,
        pt2e_quantizer=quantizer,
        backend_hardware=backend_hardware,
        alloc_graph_input=False,
        alloc_graph_output=False,
    ),
)

## 6. Save

In [ ]:
with open("hf_causal_lm_qnn.pte", "wb") as f:
    et_program_manager.write_to_file(f)
    print(f"Saved {f.name} ({f.tell() / 1e9:.2f} GB)")

## 7. Execute
Run the `hf_causal_lm_qnn.pte` on any of the target SoCs.